In [0]:
file_path = "/Volumes/workspace/default/sales_volume/sales.csv"

print("File path:", file_path)

In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

display(df)

In [0]:
# Total rows
print("Total rows:", df.count())

# Total columns
print("Total columns:", len(df.columns))

# Column names and data types
df.printSchema()

# Display first 5 records
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).display()

In [0]:
from pyspark.sql.functions import col, expr, when

clean_df = (
    df.dropDuplicates(["order_id"])
      .withColumn("customer", when(col("customer").isNull(), "Unknown").otherwise(col("customer")))
      .withColumn("city", when(col("city").isNull(), "Unknown").otherwise(col("city")))
      .withColumn("price", expr("try_cast(price AS DOUBLE)"))
      .withColumn("quantity", expr("try_cast(quantity AS INT)"))
      .filter((col("price") > 0) & (col("quantity") > 0))
)

display(clean_df)